## SilverWork Incremental Industrial v2

### Step 1 - Import and Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, UTC
import uuid

In [0]:
spark.sql("use catalog novacart_adb")
spark.sql("create schema if not exists silver_schema")

silver_run_id = str(uuid.uuid4())
print("current silver run id: ",silver_run_id)

### Step 2 - Silver Control Table
This table stores the latest silver processing state for each entity

It help us track:
- The latest Bronze run already processed by Silver
- The latest Bronze ingestion timestamp already processed
-  How many rows merged in the latest silver run

In [0]:
spark.sql("""
          create table if not exists novacart_adb.silver_schema.processing_ctrl
          (
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
          )
          using delta
          """)

### Step 3 - Helper Function
This cell contains the reusale logic for Silver:
- upsert_to_silver() - merges cleaned / transformed rows to the silver target table
- get_last_processed_bronze_ingested_at() - reads the silver watermark
- upsert_silver_control() - updates the silver control table
- get_incremental_bronze() - reads only new bronze rows that silver has not processed yet

In [0]:
def upsert_to_silver(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("trg").merge(df_source.alias("src"), f"trg.{join_key} = src.{join_key}")
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        df_source.write.format('delta').saveAsTable(target_table)

In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
    ctrl = spark.table("novacart_adb.silver_schema.processing_ctrl")\
                .filter(
                    (col("layer") == "Silver") &
                    (col("entity_name") == entity_name) &
                    (col("run_status") == "Success") 
                )\
                .orderBy(col("updated_at").desc())\
                .limit(1)

    rows = ctrl.collect()
    if not rows:
        return None
    return rows[0]["last_processed_bronze_ingested_at"]

In [0]:
def upsert_silver_ctrl(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "Success",
                silver_run_id,
                datetime.now()
            )
        ],
                schema = """
                layer string,
                entity_name string,
                last_processed_bronze_run_id string,
                last_processed_bronze_ingested_at timestamp,
                rows_merged bigint,
                run_status string,
                silver_run_id string,
                updated_at timestamp  
                """
    )

    dt = DeltaTable.forName(spark, "novacart_adb.silver_schema.processing_ctrl")
    (dt.alias("trg").merge(ctrl_df.alias("src"), "trg.layer = src.layer and trg.entity_name = src.entity_name")\
        .whenMatchedUpdate(set={
            "last_processed_bronze_run_id": "src.last_processed_bronze_run_id",
            "last_processed_bronze_ingested_at": "src.last_processed_bronze_ingested_at",
            "rows_merged": "src.rows_merged",
            "run_status": "src.run_status",
            "silver_run_id": "src.silver_run_id",
            "updated_at": "src.updated_at"
        })\
        .whenNotMatchedInsertAll()\
        .execute())

In [0]:
def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)

    bronze_df = spark.read.table(bronze_table)
    if last_ingested_at is None:
        return bronze_df, last_ingested_at
    
    return bronze_df.filter(col("bronze_ingested_at") > lit(last_ingested_at)), last_ingested_at


### Step 4 - Orders incremental processing
It processes **Orders** from bronze to silver

In [0]:
df_raw = spark.sql("select * from novacart_adb.bronze_schema.orders_raw")
display(df_raw)

In [0]:
# Reads only the Bronze order rows that Silevr has not processed yet
orders_inc, last_orders_ingested_at = get_incremental_bronze("novacart_adb.bronze_schema.orders_raw", "orders")

# Count the incremental order rows entering silver in this run
orders_inc_count = orders_inc.count()
print(f"Orders rows to be processed in silver: {orders_inc_count}")

# when new orders data available, clean & validate data
if orders_inc_count > 0:
    
    order_window = Window.partitionBy("order_id").orderBy(col("updated_at").cast("timestamp").desc(), col("bronze_ingested_at").desc())

    orders_cleaned = (
        orders_inc.withColumn("order_status", when(col("order_status") == "", lit(None)).otherwise(upper(trim(col("order_status")))))

        .withColumn("order_amount", regexp_replace(col("order_amount"), r"[$, ]",""))
        .withColumn("order_amount", when(trim(col("order_amount")).isin("N/A", "NULL", "??", ""),None).otherwise(col("order_amount")))
        .withColumn("order_amount", col("order_amount").cast("double"))

        .withColumn("created_at", to_timestamp("created_at"))
        .withColumn("updated_at", to_timestamp("updated_at"))

        .withColumn("row_rank", row_number().over(order_window))
        .filter(col("row_rank") == 1).drop("row_rank")
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    # Merge cleaned Silver dataset into Delta target table
    upsert_to_silver(orders_cleaned, "novacart_adb.silver_schema.orders_cleaned", "order_id")

    # Apply Silver data quality rules to cleaned order records
    orders_validated = (
        orders_cleaned
        .withColumn("to_be_verified_by_orders_team",
                    when(col("customer_id").isNull(), "verify_customer_id")
                    .when(col("product_id").isNull(), "verify_product_id")
                    .when(col("order_status").isNull() | (trim(col("order_status")) == ""), "verify_order_status")
                    .when(col("order_amount").isNull() | (col("order_amount") <= 0), "verify_order_amount")
                    .otherwise("No Issues")
                    )
        .withColumn(
            "check_order_amount",
            when(col("order_amount").isNull() | (col("order_amount") <= 0), lit(True)).otherwise(lit(False))
        )
        .withColumn("order_date", to_date("created_at"))
        .withColumn("order_year", year("created_at"))
        .withColumn("order_month", month("created_at"))
        .withColumn("order_day", dayofmonth("created_at"))
        .withColumn("order_dow", date_format("created_at", "E"))
    )

    #Keep only the valid data rows for transformed silver table
    orders_good = orders_validated.filter(col("to_be_verified_by_orders_team") == "No Issues")

    # Send invalid rows to the quarantine dataset for manual review
    orders_bad = orders_validated.filter(col("to_be_verified_by_orders_team") != "No Issues")\
                                    .withColumn("quarantine_ts", current_timestamp())

    # Merge validated Silver dataset into Delta target table
    upsert_to_silver(orders_good, "novacart_adb.silver_schema.orders_transformed", "order_id")

    # Append bad order rows to quarantine table
    orders_bad.write.format('delta').mode("append").saveAsTable("novacart_adb.silver_schema.orders_quarantine")

    max_ingested = orders_inc.agg(max(col("bronze_ingested_at")).alias("mx")).collect()[0]['mx']

    max_run_id = orders_inc.filter(col("bronze_ingested_at") == lit(max_ingested)).agg(max(col("bronze_run_id")).alias('mx')).collect()[0]['mx']

    upsert_silver_ctrl("orders", max_run_id, max_ingested, orders_good.count())

else:
    print("No new orders data available for Silver")

    upsert_silver_ctrl("orders", None, last_orders_ingested_at, orders_inc_count)



In [0]:
print(spark.sql("select * from novacart_adb.silver_schema.orders_cleaned").count())
print(spark.sql("select * from novacart_adb.silver_schema.orders_quarantine").count())
print(spark.sql("select * from novacart_adb.silver_schema.orders_transformed").count())
display(spark.sql("select * from novacart_adb.silver_schema.processing_ctrl"))

In [0]:
%sql
select distinct(order_id), * from novacart_adb.silver_schema.orders_quarantine

### Products Incremental Processing

In [0]:
%sql
select * from novacart_adb.bronze_schema.products_raw;

In [0]:
# Reads only the Bronze product rows that Silevr has not processed yet
products_inc, last_products_ingested_at = get_incremental_bronze("novacart_adb.bronze_schema.products_raw", "products")

# Count the incremental product rows entering silver in this run
products_inc_count = products_inc.count()
print(f"products rows to be processed in silver: {products_inc_count}")

# when new products data available, clean & validate data
if products_inc_count > 0:
    
    product_window = Window.partitionBy("product_id").orderBy(col("updated_at").cast("timestamp").desc(), col("bronze_ingested_at").desc())

    products_cleaned = (
        products_inc.withColumn("product_name", when(col("product_name") == "", lit(None)).otherwise(upper(trim(col("product_name")))))

        .withColumn("category", when(upper(trim(col("category"))).contains("ELECTRNICS"),"ELECTRONICS").otherwise(upper(trim(col("category")))))

        .withColumn("price", upper(trim(col("price"))))
        .withColumn("price", regexp_replace(col("price"), r"\$",""))
        .withColumn("price", regexp_replace(col("price"), ",","."))
        .withColumn("price", regexp_replace(col("price"), r"\s+",""))
        .withColumn("price", expr("try_cast(price as double)"))
        .withColumn("updated_at", to_timestamp("updated_at"))
 
        .withColumn("row_rank", row_number().over(product_window))
        .filter(col("row_rank") == 1).drop("row_rank")
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    # Merge cleaned Silver dataset into Delta target table
    upsert_to_silver(products_cleaned, "novacart_adb.silver_schema.products_cleaned", "product_id")

    # Apply Silver data quality rules to cleaned product records
    products_validated = (
        products_cleaned
        .withColumn("to_be_verified_by_products_team",
                    when(col("product_name").isNull(), "verify_product_name")
                    .when(col("category").isNull(), "verify_category")
                    .when(col("price").isNull() | (col("price") <= 0), "verify_price")
                    .otherwise("No Issues")
                    )
        .withColumn(
            "check_product_price",
            when(col("price").isNull() | (col("price") <= 0), "invalid_price").otherwise("valid_price")
        )
    )

    #Keep only the valid data rows for transformed silver table
    products_good = products_validated.filter(
        (col("to_be_verified_by_products_team") == "No Issues") & 
        (col("check_product_price") == "valid_price")
    )
    if "price_raw" in products_good.columns:
        products_good = products_good.drop("price_raw")
    # Send invalid rows to the quarantine dataset for manual review
    products_bad = products_validated.filter(
        (col("to_be_verified_by_products_team") != "No Issues") | 
        (col("check_product_price") == "invalid_price")).withColumn("quarantine_ts", current_timestamp())

    # Merge validated Silver dataset into Delta target table
    upsert_to_silver(products_good, "novacart_adb.silver_schema.products_transformed", "product_id")

    # Append bad product rows to quarantine table
    products_bad.write.format('delta').mode("append").saveAsTable("novacart_adb.silver_schema.products_quarantine")

    max_ingested = products_inc.agg(max(col("bronze_ingested_at")).alias("mx")).collect()[0]['mx']

    max_run_id = products_inc.filter(col("bronze_ingested_at") == lit(max_ingested)).agg(max(col("bronze_run_id")).alias('mx')).collect()[0]['mx']

    upsert_silver_ctrl("products", max_run_id, max_ingested, products_good.count())

else:
    print("No new products data available for Silver")

    upsert_silver_ctrl("products", None, last_products_ingested_at, products_inc_count)



In [0]:
print(spark.sql("select * from novacart_adb.silver_schema.products_cleaned").count())
print(spark.sql("select * from novacart_adb.silver_schema.products_quarantine").count())
print(spark.sql("select * from novacart_adb.silver_schema.products_transformed").count())
display(spark.sql("select * from novacart_adb.silver_schema.processing_ctrl"))

In [0]:
%sql
select * from novacart_adb.silver_schema.products_quarantine

### Step 6 - Payments Incremental Processing

In [0]:
# Reads only the Bronze payment rows that Silevr has not processed yet
payments_inc, last_payments_ingested_at = get_incremental_bronze("novacart_adb.bronze_schema.payments_raw", "payments")

# Count the incremental payment rows entering silver in this run
payments_inc_count = payments_inc.count()
print(f"payments rows to be processed in silver: {payments_inc_count}")

# when new payments data available, clean & validate data
if payments_inc_count > 0:
    
    payment_window = Window.partitionBy("payment_id").orderBy(col("processed_at").cast("timestamp").desc(), col("bronze_ingested_at").desc())

    payments_cleaned = (
        payments_inc.withColumn("payment_status", when(col("payment_status") == "", lit(None)).otherwise(upper(trim(col("payment_status")))))

        .withColumn("paid_amount", upper(trim(col("paid_amount"))))
        .withColumn("paid_amount", regexp_replace(col("paid_amount"), r"\$",""))
        .withColumn("paid_amount", regexp_replace(col("paid_amount"), ",","."))
        .withColumn("paid_amount", regexp_replace(col("paid_amount"), r"\s+",""))
        .withColumn("paid_amount", expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at", to_timestamp("processed_at"))
 
        .withColumn("row_rank", row_number().over(payment_window))
        .filter(col("row_rank") == 1).drop("row_rank")
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    # Merge cleaned Silver dataset into Delta target table
    upsert_to_silver(payments_cleaned, "novacart_adb.silver_schema.payments_cleaned", "payment_id")

    # Apply Silver data quality rules to cleaned payment records
    payments_validated = (
        payments_cleaned
        .withColumn("to_be_verified_by_payments_team",
                    when(col("order_id").isNull(), "verify_order_id")
                    .when(col("payment_status").isNull(), "verify_payment_status")
                    .when(col("paid_amount").isNull() | (col("paid_amount") <= 0), "verify_paid_amount")
                    .otherwise("No Issues")
                    )
        .withColumn(
            "check_paid_amount",
            when(col("paid_amount").isNull() | (col("paid_amount") <= 0), lit(True)).otherwise(lit(False))
        )
    )

    #Keep only the valid data rows for transformed silver table
    payments_good = payments_validated.filter((col("to_be_verified_by_payments_team") == "No Issues"))
    
    # Send invalid rows to the quarantine dataset for manual review
    payments_bad = payments_validated.filter((col("to_be_verified_by_payments_team") != "No Issues")).withColumn("quarantine_ts", current_timestamp())

    # Merge validated Silver dataset into Delta target table
    upsert_to_silver(payments_good, "novacart_adb.silver_schema.payments_transformed", "payment_id")

    # Append bad payment rows to quarantine table
    payments_bad.write.format('delta').mode("append").saveAsTable("novacart_adb.silver_schema.payments_quarantine")

    max_ingested = payments_inc.agg(max(col("bronze_ingested_at")).alias("mx")).collect()[0]['mx']

    max_run_id = payments_inc.filter(col("bronze_ingested_at") == lit(max_ingested)).agg(max(col("bronze_run_id")).alias('mx')).collect()[0]['mx']

    upsert_silver_ctrl("payments", max_run_id, max_ingested, payments_good.count())

else:
    print("No new payments data available for Silver")

    upsert_silver_ctrl("payments", None, last_payments_ingested_at, payments_inc_count)



In [0]:
print(spark.sql("select * from novacart_adb.silver_schema.payments_cleaned").count())
print(spark.sql("select * from novacart_adb.silver_schema.payments_quarantine").count())
print(spark.sql("select * from novacart_adb.silver_schema.payments_transformed").count())
display(spark.sql("select * from novacart_adb.silver_schema.processing_ctrl"))
display(spark.sql("select * from novacart_adb.bronze_schema.ingestion_control"))